# OC14 — SFT (LoRA) · Qwen3-1.7B-Instruct · medical-triage assistant

Supervised fine-tuning with Unsloth + LoRA on Kaggle (T4). Reads the private dataset
`oc14-triage-data`. **First run: `SMOKE = True`** → 5 steps (~minutes) to validate the
pipeline and library versions. Then set `SMOKE = False` and re-run for the full ~2 epochs.

Optional Kaggle **Secrets** (Add-ons → Secrets): `WANDB_API_KEY` (loss curves),
`HF_TOKEN` (push the adapter). Without them the notebook still runs and saves the adapter
to the notebook output.

In [ ]:
# Official Unsloth Kaggle install (verbatim, June 2026): install torch from the cu128
# index FIRST so its wheels carry the T4/sm_75 CUDA kernels. Plain `pip install unsloth`
# let pip resolve a torch build lacking them -> 'CUDA: no kernel image' at model load.
!pip install pip3-autoremove
!pip install torch torchvision torchaudio xformers --index-url https://download.pytorch.org/whl/cu128
!pip install unsloth
!pip install --no-deps --upgrade "torchao>=0.16.0"
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

In [ ]:
import subprocess, sys
with open('/kaggle/working/requirements-train.lock.txt', 'w') as fh:
    subprocess.run([sys.executable, '-m', 'pip', 'freeze'], stdout=fh)
print('lockfile written ->', '/kaggle/working/requirements-train.lock.txt')

In [ ]:
import os
try:
    from kaggle_secrets import UserSecretsClient
    _us = UserSecretsClient()
    for _k in ('WANDB_API_KEY', 'HF_TOKEN'):
        try:
            os.environ[_k] = _us.get_secret(_k)
        except Exception:
            pass
except Exception:
    pass
REPORT_TO = 'wandb' if os.environ.get('WANDB_API_KEY') else 'none'
HF_TOKEN = os.environ.get('HF_TOKEN')
print('W&B:', REPORT_TO, '| HF push:', bool(HF_TOKEN))

In [ ]:
# ---- config ----
SMOKE = False              # full run (~2 epochs). Set True for a ~5-step validation (minutes).
MODEL = 'Qwen/Qwen3-1.7B'
MAX_SEQ_LEN = 2048         # covers ~99.6% of SFT rows (p99 ~1296 tok); rare longer rows truncated
SEED = 3407
DATA_DIR = '/kaggle/input/oc14-triage-data'
OUT = '/kaggle/working/sft_adapter'
HF_REPO = 'ghislaindelabie/oc14-qwen3-1.7b-instruct-sft-lora'

In [ ]:
from unsloth import FastLanguageModel
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL, max_seq_length=MAX_SEQ_LEN,
    load_in_4bit=True, load_in_8bit=False, full_finetuning=False)
model = FastLanguageModel.get_peft_model(
    model, r=16, lora_alpha=16, lora_dropout=0, bias='none',
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    use_gradient_checkpointing='unsloth', random_state=SEED,
    use_rslora=False, loftq_config=None)
print('eos_token:', tokenizer.eos_token, '| id:', tokenizer.eos_token_id)  # expect <|endoftext|>

In [ ]:
import glob
# Qwen3-1.7B-Instruct ships NO chat_template -> set a plain ChatML one (Qwen-style, no <think>).
# This propagates to the saved tokenizer, so vLLM serving uses the same format later.
tokenizer.chat_template = (
    "{% for message in messages %}"
    "{{'<|im_start|>' + message['role'] + '\n' + message['content'] + '<|im_end|>' + '\n'}}"
    "{% endfor %}"
    "{% if add_generation_prompt %}{{'<|im_start|>assistant\n'}}{% endif %}")
print('inputs:', os.listdir('/kaggle/input') if os.path.isdir('/kaggle/input') else 'NONE')
_hits = glob.glob('/kaggle/input/**/sft_train.jsonl', recursive=True)
assert _hits, 'sft_train.jsonl not found under /kaggle/input — is the dataset attached?'
DATA_DIR = os.path.dirname(_hits[0])
print('DATA_DIR =', DATA_DIR)
from datasets import load_dataset
ds = load_dataset('json', data_files={
    'train': f'{DATA_DIR}/sft_train.jsonl', 'val': f'{DATA_DIR}/sft_val.jsonl'})
def to_text(ex):
    return {'text': tokenizer.apply_chat_template(ex['messages'], tokenize=False,
                                                  add_generation_prompt=False)}
ds = ds.map(to_text, remove_columns=ds['train'].column_names)
print(ds)
print(ds['train'][0]['text'][:500])

In [ ]:
from trl import SFTConfig, SFTTrainer
args = SFTConfig(
    dataset_text_field='text',
    per_device_train_batch_size=4, gradient_accumulation_steps=8,
    warmup_ratio=0.05, num_train_epochs=2, max_steps=(5 if SMOKE else -1),
    learning_rate=2e-4, logging_steps=5, save_steps=50, save_total_limit=2,
    optim='adamw_8bit', weight_decay=0.01, lr_scheduler_type='linear',
    seed=SEED, output_dir='/kaggle/working/sft_out', report_to=REPORT_TO,
    run_name='oc14-sft-qwen3-instruct', padding_free=False)
trainer = SFTTrainer(model=model, tokenizer=tokenizer,
                     train_dataset=ds['train'], eval_dataset=ds['val'], args=args)
# Loss on assistant turns only (mask system/user) — Qwen3 ChatML markers.
from unsloth.chat_templates import train_on_responses_only
trainer = train_on_responses_only(
    trainer, instruction_part='<|im_start|>user\n', response_part='<|im_start|>assistant\n')

In [ ]:
import time
_t = time.time()
stats = trainer.train()
print('train seconds:', round(time.time() - _t, 1))
print(getattr(stats, 'metrics', stats))

In [ ]:
model.save_pretrained(OUT)
tokenizer.save_pretrained(OUT)
print('saved adapter ->', OUT)
if HF_TOKEN and not SMOKE:
    model.push_to_hub(HF_REPO, token=HF_TOKEN)
    tokenizer.push_to_hub(HF_REPO, token=HF_TOKEN)
    print('pushed adapter ->', HF_REPO)

In [ ]:
# Eyeball a couple of triage answers (rough after a SMOKE run — just a sanity check).
FastLanguageModel.for_inference(model)
SYS = 'Tu es un assistant de triage médical du CHSA. Donne le niveau d\'urgence, une justification et une recommandation.'
for q in ['Un patient de 60 ans a une douleur thoracique aiguë avec sueurs depuis 20 min.',
          'A 30-year-old has mild seasonal allergies and itchy eyes. What do you advise?']:
    msgs = [{'role': 'system', 'content': SYS}, {'role': 'user', 'content': q}]
    ids = tokenizer.apply_chat_template(msgs, add_generation_prompt=True, return_tensors='pt'
                                        ).to(model.device)
    out = model.generate(input_ids=ids, max_new_tokens=200, temperature=0.7, do_sample=True)
    print('Q:', q)
    print(tokenizer.decode(out[0][ids.shape[1]:], skip_special_tokens=True))
    print('-' * 70)